# M-Shots Learning

In this notebook, we'll explore small prompt engineering techniques and recommendations that will help us elicit responses from the models that are better suited to our needs.

In [11]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

# Formatting the answer with Few Shot Samples.

To obtain the model's response in a specific format, we have various options, but one of the most convenient is to use Few-Shot Samples. This involves presenting the model with pairs of user queries and example responses.

Large models like GPT-3.5 respond well to the examples provided, adapting their response to the specified format.

Depending on the number of examples given, this technique can be referred to as:
* Zero-Shot.
* One-Shot.
* Few-Shots.

With One Shot should be enough, and it is recommended to use a maximum of six shots. It's important to remember that this information is passed in each query and occupies space in the input prompt.



In [12]:
# Function to call the model.
def return_OAIResponse(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=1,
        )

    return (response.choices[0].message.content)

In this zero-shots prompt we obtain a correct response, but without formatting, as the model incorporates the information he wants.

In [13]:
#zero-shot
context_user = [
    {'role':'system', 'content':'You are an expert in F1.'}
]
print(return_OAIResponse("Who won the F1 2010?", context_user))

Sebastian Vettel won the F1 World Championship in 2010. He drove for Red Bull Racing and became the youngest Formula 1 World Champion at that time.


For a model as large and good as GPT 3.5, a single shot is enough to learn the output format we expect.


In [14]:
#one-shot
context_user = [
    {'role':'system', 'content':
     """You are an expert in F1.

     Who won the 2000 f1 championship?
     Driver: Michael Schumacher.
     Team: Ferrari."""}
]
print(return_OAIResponse("Who won the F1 2011?", context_user))

Driver: Sebastian Vettel.
Team: Red Bull Racing.


Smaller models, or more complicated formats, may require more than one shot. Here a sample with two shots.

In [15]:
#Few shots
context_user = [
    {'role':'system', 'content':
     """You are an expert in F1.

     Who won the 2010 f1 championship?
     Driver: Sebastian Vettel.
     Team: Red Bull Renault.

     Who won the 2009 f1 championship?
     Driver: Jenson Button.
     Team: BrawnGP."""}
]
print(return_OAIResponse("Who won the F1 2006?", context_user))

Driver: Fernando Alonso.
Team: Renault.


In [16]:
print(return_OAIResponse("Who won the F1 2019?", context_user))

The 2019 F1 championship was won by Lewis Hamilton from Mercedes.


We've been creating the prompt without using OpenAI's roles, and as we've seen, it worked correctly.

However, the proper way to do this is by using these roles to construct the prompt, making the model's learning process even more effective.

By not feeding it the entire prompt as if they were system commands, we enable the model to learn from a conversation, which is more realistic for it.

In [17]:
#Recomended solution
context_user = [
    {'role':'system', 'content':'You are and expert in f1.\n\n'},
    {'role':'user', 'content':'Who won the 2010 f1 championship?'},
    {'role':'assistant', 'content':"""Driver: Sebastian Vettel. \nTeam: Red Bull. \nPoints: 256. """},
    {'role':'user', 'content':'Who won the 2009 f1 championship?'},
    {'role':'assistant', 'content':"""Driver: Jenson Button. \nTeam: BrawnGP. \nPoints: 95. """},
]

print(return_OAIResponse("Who won the F1 2019?", context_user))

Driver: Lewis Hamilton. 
Team: Mercedes. 
Points: 413.


We could also address it by using a more conventional prompt, describing what we want and how we want the format.

However, it's essential to understand that in this case, the model is following instructions, whereas in the case of use shots, it is learning in real-time during inference.

In [18]:
context_user = [
    {'role':'system', 'content':"""You are and expert in f1.
    You are going to answer the question of the user giving the name of the rider,
    the name of the team and the points of the champion, following the format:
    Drive:
    Team:
    Points: """
    }
]

print(return_OAIResponse("Who won the F1 2019?", context_user))

Driver: Lewis Hamilton
Team: Mercedes
Points: 413


In [19]:
context_user = [
    {'role':'system', 'content':
     """You are classifying .

     Who won the 2010 f1 championship?
     Driver: Sebastian Vettel.
     Team: Red Bull Renault.

     Who won the 2009 f1 championship?
     Driver: Jenson Button.
     Team: BrawnGP."""}
]
print(return_OAIResponse("Who won the F1 2006?", context_user))

Sorry, I do not have that information.


Few Shots for classification.


In [20]:
context_user = [
    {'role':'system', 'content':
     """You are an expert in reviewing product opinions and classifying them as positive or negative.

     It fulfilled its function perfectly, I think the price is fair, I would buy it again.
     Sentiment: Positive

     It didn't work bad, but I wouldn't buy it again, maybe it's a bit expensive for what it does.
     Sentiment: Negative.

     I wouldn't know what to say, my son uses it, but he doesn't love it.
     Sentiment: Neutral
     """}
]
print(return_OAIResponse("I'm not going to return it, but I don't plan to buy it again.", context_user))

Sentiment: Negative


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

## Exercise Answer

For this exercise I tried three different prompt versions using the idea of zero-shot, one-shot, and few-shot prompting. The goal was to see how examples change the model output.


### Version 1 - Zero-Shot Classification

In [21]:
# In this version, I only explain the task. I do not give the model any examples.


context_user = [
    {
        'role': 'system',
        'content': '''You classify student feedback as Positive, Negative, or Neutral.
        Answer only with the sentiment label.'''
    }
]

print(return_OAIResponse("The class was useful, but sometimes the pace was too fast.", context_user))


#Expected behavior: the model should probably answer `Neutral`, because the sentence has both good and bad parts.

Neutral


### Version 2 - One-Shot Classification


In [ ]:
# Here I give one example before asking the real question. 

context_user = [
    {
        'role': 'system',
        'content': '''You classify student feedback as Positive, Negative, or Neutral.

        Feedback: The teacher explained everything clearly and I learned a lot.
        Sentiment: Positive
        '''
    }
]

print(return_OAIResponse("The class was useful, but sometimes the pace was too fast.", context_user))


# Expected behavior: the model should answer in the same format, like `Sentiment: Neutral`.


Sentiment: Neutral



### Version 3 - Few-Shot Classification

In [ ]:

# This version gives multiple examples. 
context_user = [
    {
        'role': 'system',
        'content': '''You classify student feedback as Positive, Negative, or Neutral.

        Feedback: The teacher explained everything clearly and I learned a lot.
        Sentiment: Positive

        Feedback: The notebook was confusing and I could not finish the lab.
        Sentiment: Negative

        Feedback: The class was okay, but I still need more practice.
        Sentiment: Neutral
        '''
    }
]

print(return_OAIResponse("The class was useful, but sometimes the pace was too fast.", context_user))


# Expected behavior: this should be the most consistent version. The model has seen examples of all three labels, so it has less need to guess.


Sentiment: Neutral


### Creative Version - Restaurant Intent Classifier

In [ ]:

context_user = [
    {
        'role': 'system',
        'content': '''Classify the customer message into one of these intents:
        Order, Complaint, Reservation, Question, or Other.

        Customer: I want two large pepperoni pizzas.
        Intent: Order

        Customer: My fries were cold when they arrived.
        Intent: Complaint

        Customer: Can I book a table for four tonight?
        Intent: Reservation
        '''
    }
]

print(return_OAIResponse("Do you have vegan options?", context_user))

# Expected behavior: the answer should be `Intent: Question`.

Intent: Question




## Report

In this lab I learned that m-shot prompting is about teaching the model the answer pattern by giving examples inside the prompt. Zero-shot means we ask directly with no examples. One-shot means we give one example. Few-shot means we give multiple examples. The examples are not training the model permanently, but they guide the model during that one API call.

The zero-shot version can work when the task is simple, but the answer format may not be consistent. For example, the model might answer Neutral, or it might write a full sentence explaining why. That is not wrong, but it may not be what we want if we need a clean label.

The one-shot version is better because the model sees the format once. If the example says Sentiment: Positive, the model is more likely to answer with Sentiment: Neutral, instead of writing a paragraph. This is useful when we care about formatting.

The few-shot version worked best in my opinion because it gave examples for Positive, Negative, and Neutral. This makes the task clearer and reduces guessing. It is especially useful when the categories are similar or when the model needs to follow a specific structure.

One version that could fail is a prompt with bad or incomplete examples. For example, if I only give positive examples, the model may become biased toward positive answers. Also, if the examples do not match the real question, the model can hallucinate or choose the wrong format. Few-shot prompting is powerful, but the examples need to be good.

My main takeaway is that examples inside the prompt can control both the content and the format of the model answer. The system message gives the overall task, but the examples show the model exactly how the answer should look. I should use zero-shot for simple questions, one-shot when I need a basic format, and few-shot when I need more reliable structure or classification.
